# Two-Ticker Baseline: Lag Feature Ablation

This notebook tests whether explicit lagged features improve 1-day direction prediction on the same `TSLA + AAPL` panel.

Training modes:
- `pooled`
- `separate`

Feature variants:
- `base`
- `base_plus_price_lags`
- `base_plus_alt_lags`
- `base_plus_all_lags`

Lag steps:
- `1D`
- `2D`
- `3D`
- `5D`

Models:
- `logreg`
- `random_forest`

Artifacts are written to `notebooks_two_tickers/outputs/lag_feature_ablation`.


In [1]:
import pandas as pd

from lag_feature_ablation_experiment import run_lag_feature_ablation_experiment

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 260)


In [2]:
result = run_lag_feature_ablation_experiment()

print(f"Dataset: {result['dataset_path']}")
print(f"Metrics: {result['metrics_path']}")
print(f"Predictions: {result['predictions_path']}")
print(f"Metadata: {result['metadata_path']}")
print(f"Feature variants: {result['variants_path']}")


Dataset: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\datasets\stock_direction_two_tickers_base.csv
Metrics: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\lag_feature_ablation\metrics.csv
Predictions: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\lag_feature_ablation\predictions.csv
Metadata: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\lag_feature_ablation\run_metadata.json
Feature variants: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks_two_tickers\outputs\lag_feature_ablation\feature_variants_by_mode.json


In [3]:
test_all = result["metrics"].query("split == 'test' and ticker == 'ALL'").copy()
test_all = test_all.sort_values(["training_mode", "model_name", "feature_variant"]).reset_index(drop=True)
test_all


,training_mode,model_name,feature_variant,split,ticker,n_rows,n_features,accuracy,balanced_accuracy,f1,roc_auc
0,pooled,logreg,base,test,ALL,256,33,0.476562,0.482137,0.464000,0.490930
1,pooled,logreg,base_plus_all_lags,test,ALL,256,153,0.578125,0.581074,0.584615,0.577261
2,pooled,logreg,base_plus_alt_lags,test,ALL,256,105,0.531250,0.535203,0.531250,0.531206
3,pooled,logreg,base_plus_price_lags,test,ALL,256,81,0.488281,0.491576,0.490272,0.542089
4,pooled,random_forest,base,test,ALL,256,33,0.578125,0.574310,0.614286,0.547132
5,pooled,random_forest,base_plus_all_lags,test,ALL,256,153,0.542969,0.534495,0.600683,0.556785
6,pooled,random_forest,base_plus_alt_lags,test,ALL,256,105,0.574219,0.563949,0.635452,0.554326
7,pooled,random_forest,base_plus_price_lags,test,ALL,256,81,0.515625,0.514050,0.544118,0.541720
8,separate,logreg,base,test,ALL,256,32,0.492188,0.496526,0.488189,0.510115
9,separate,logreg,base_plus_all_lags,test,ALL,256,152,0.519531,0.521706,0.528736,0.556724


In [4]:
summary = test_all.pivot_table(
    index=["training_mode", "model_name"],
    columns="feature_variant",
    values="balanced_accuracy",
)
summary


feature_variant                  base  base_plus_all_lags  base_plus_alt_lags  base_plus_price_lags
training_mode model_name                                                                           
pooled        logreg         0.482137            0.581074            0.535203              0.491576
              random_forest  0.574310            0.534495            0.563949              0.514050
separate      logreg         0.496526            0.521706            0.498094              0.513835
              random_forest  0.533850            0.556324            0.542151              0.552973

In [5]:
test_by_ticker = result["metrics"].query("split == 'test' and ticker != 'ALL'").copy()
test_by_ticker = test_by_ticker.sort_values(["ticker", "training_mode", "model_name", "feature_variant"]).reset_index(drop=True)
test_by_ticker


,training_mode,model_name,feature_variant,split,ticker,n_rows,n_features,accuracy,balanced_accuracy,f1,roc_auc
0,pooled,logreg,base,test,AAPL,128,33,0.484375,0.481281,0.521739,0.482759
1,pooled,logreg,base_plus_all_lags,test,AAPL,128,153,0.617188,0.614532,0.647482,0.644335
2,pooled,logreg,base_plus_alt_lags,test,AAPL,128,105,0.585938,0.583005,0.618705,0.611084
3,pooled,logreg,base_plus_price_lags,test,AAPL,128,81,0.492188,0.486946,0.539007,0.535222
4,pooled,random_forest,base,test,AAPL,128,33,0.562500,0.552709,0.621622,0.552463
5,pooled,random_forest,base_plus_all_lags,test,AAPL,128,153,0.593750,0.575369,0.675000,0.567734
6,pooled,random_forest,base_plus_alt_lags,test,AAPL,128,105,0.585938,0.563793,0.678788,0.580788
7,pooled,random_forest,base_plus_price_lags,test,AAPL,128,81,0.531250,0.518227,0.605263,0.566256
8,separate,logreg,base,test,AAPL,128,32,0.500000,0.508867,0.475410,0.511576
9,separate,logreg,base_plus_all_lags,test,AAPL,128,152,0.546875,0.551724,0.546875,0.615764
